In [16]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, accuracy_score
from sklearn.pipeline import Pipeline
import joblib
import re
from typing import Tuple, List, Dict
import spacy
from collections import Counter
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import sys
from pathlib import Path
import pandas as pd
from sklearn.utils import resample
from collections import Counter
from scipy.sparse import hstack

sys.path.append(str(Path.cwd().parents[1]))
from Modelling.preprocessing import cleanTextPipeline

In [17]:
nltk.download("stopwords")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\swoye\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [18]:
stop_words = set(stopwords.words('english'))

In [ ]:
class lightWeightIntentClassifier:
    def __init__(self, method = 'Hybrid'):
        self.method = method
        self.model = None
        self.vectorizer = None
        self.feature_weights = {}

        try:
            self.nlp = spacy.load("en_core_web_sm")
            self.use_spacy = True
        except:
            self.use_spacy = False
            print("Unable to use space, using simple preprocessing")

    def advancedPreprocessing(self, text: str) -> str:
        text = text.lower().strip()

        important_words = {"what", "why", "how", "when", "where", "who", "which",
                   "i", "me", "my", "mine", "we", "our", "us"}
        
        for word in important_words:
            self.nlp.vocab[word].is_stop = False

        if self.use_spacy == True:
            doc = self.nlp(text)

            tokens = [token.lemma_ for token in doc if not token.is_stop and not token.is_punct]

            return ' '.join(tokens)    
        else:
            final_text = cleanTextPipeline(text)
            return final_text
        
    def extract_advanced_features(self, text: str) -> Dict[str, int]:
        """Extract domain-specific features for finance queries"""
        text_lower = text.lower()
        features = {}
        
        personal_pronouns = ['i', 'my', 'me', 'mine', 'myself']
        features['personal_pronoun_count'] = sum(1 for pronoun in personal_pronouns 
                                                if pronoun in text_lower)
        
        time_refs = ['month', 'week', 'year', 'today', 'yesterday', 'last', 'this']
        features['time_reference_count'] = sum(1 for ref in time_refs 
                                             if ref in text_lower)
        
        question_words = ['what', 'how', 'why', 'when', 'where', 'explain', 'define']
        features['question_word_count'] = sum(1 for word in question_words 
                                            if word in text_lower)
        
        personal_actions = ['spend', 'spent', 'buy', 'bought', 'paid', 'save', 'saved']
        features['personal_action_count'] = sum(1 for action in personal_actions 
                                              if action in text_lower)
        
        concept_words = ['interest', 'investment', 'stock', 'bond', 'fund', 'market']
        features['concept_word_count'] = sum(1 for concept in concept_words 
                                           if concept in text_lower)
        
        features['query_length'] = len(text.split())
        
        features['has_possessive'] = 1 if any(word in text_lower for word in ['my', 'mine']) else 0
        
        return features
    
    def create_enhanced_training_data(self):
        """Create comprehensive but focused training dataset"""
        
        personal_queries = [ # Spending patterns 
            "How much did I spend on groceries this month?", "What was my biggest expense last week?", "Show me my food spending for March", "How much money did I spend on entertainment?", "What did I buy at the mall yesterday?", "My credit card bill is how much?", "How much did I spend on gas this month?", "What were my shopping expenses last quarter?", "Show me my utility bills", "How much did I pay for Netflix?", "Am I over budget this month?", "How much is left in my grocery budget?", "What's my budget status for dining out?", "Did I stick to my monthly budget?", "How much can I still spend on clothes?", "My savings goal progress", "Budget vs actual spending comparison", "How much should I budget for vacation?", "What's my emergency fund balance?", "Am I saving enough each month?", "Show my recent bank transactions", "What charges appeared on my card?", "List my last 5 purchases", "Where did I spend money on Friday?", "My account balance right now", "What subscriptions charged me?", "Show me ATM withdrawals", "Which stores did I shop at?", "My salary deposit date", "What's pending in my account?", "How much did I earn last month?", "My income vs expenses ratio", "What's my take-home pay?", "How much tax did I pay?", "My freelance income total", "Bonus payment received when?", "Investment returns this year", "Rental income tracking", "Side hustle earnings", "My net worth calculation", "Check my grocery spending this week", "My Netflix bill amount", "How much did I spend last month?", "Show my salary deposit date", "What is left in my budget for March?", "List my last 10 purchases", "My bank account balance now", "How much did I spend on entertainment?", "What is my total income this month?", "Check pending transactions in my account" ]
        
        general_queries = [
            # Basic concepts
            "What is compound interest?",
            "How does inflation affect savings?",
            "Explain what is SIP investment",
            "What are mutual funds?",
            "How do stock markets work?",
            "What is a credit score?",
            "Define financial planning",
            "What is return on investment?",
            "How to calculate EMI?",
            "What are government bonds?",
            
            # Investment advice
            "Best investment options for beginners",
            "How to diversify investment portfolio?",
            "What is risk assessment in investing?",
            "Difference between stocks and bonds",
            "How to choose mutual funds?",
            "What is dollar cost averaging?",
            "Benefits of long-term investing",
            "How to invest in index funds?",
            "What are ETFs and how they work?",
            "Real estate vs stock investment",
            
            # Financial planning
            "How to create a budget plan?",
            "What is emergency fund planning?",
            "Retirement planning strategies",
            "How to improve credit score?",
            "Debt consolidation methods",
            "Tax saving investment options",
            "How to set financial goals?",
            "What is financial literacy?",
            "Insurance planning basics",
            "Estate planning fundamentals",
            
            # Market concepts
            "What causes stock market crashes?",
            "How do interest rates affect economy?",
            "What is market volatility?",
            "Bull market vs bear market",
            "How to read financial statements?",
            "What are dividend yields?",
            "How currency exchange rates work?",
            "What is cryptocurrency basics?",
            "Economic indicators to watch",
            "How recession affects investments?",
            "Explain mutual funds in simple terms",
            "How does diversification reduce risk?",
            "Steps to create a budget plan",
            "What is inflation and its effects?",
            "How do interest rates work?",
            "Define dollar cost averaging",
            "ETF basics for beginners",
            "How to calculate ROI",
            "What are government bonds used for?",
            "Difference between long-term and short-term investment"
        ]
        
        X = personal_queries + general_queries
        y = ['personal'] * len(personal_queries) + ['general'] * len(general_queries)
        
        return X, y
    
    def train_model(self, test_size = 0.2):
        X, y = self.create_enhanced_training_data()

        X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, test_size=test_size, stratify=y)

        if self.method == "naive_bayes":
            self.model = Pipeline([
                ('tfidf', TfidfVectorizer(
                    max_features=2000,
                    ngram_range=(1, 3),
                    min_df=2,
                    max_df=0.8
                )),
                ('nb', MultinomialNB(alpha=0.1))
            ])
        
        elif self.method == "random_forest":
            self.model = Pipeline([
                ('tfidf', TfidfVectorizer(
                    max_features=1500,
                    ngram_range=(1, 2),
                    min_df=2
                )),
                ('rf', RandomForestClassifier(
                    max_depth=10, 
                    n_estimators=50,
                    random_state=42,
                    n_jobs=-1
                ))
            ])
        
        else:

            vectorizer = TfidfVectorizer(
                max_features=1500,
                ngram_range=(1, 3),
                min_df=2,
                max_df=0.9
            )

            train_X_vector = vectorizer.fit_transform(self.advancedPreprocessing(x) for x in X_train)
            test_X_vector = vectorizer.transform(self.advancedPreprocessing(x) for x in X_test)

            ensemble = VotingClassifier(
                estimators=[("nb", MultinomialNB(alpha = 0.1)), ("rf", RandomForestClassifier(n_estimators=30, max_depth=8, random_state=42))],
                voting = "soft"
            )

            ensemble.fit(train_X_vector, y_train)

            self.vectorizer = vectorizer
            self.model = ensemble

            y_pred = ensemble.predict(test_X_vector)

            accuracy = accuracy_score(y_test, y_pred)
            # print(f"Accuracy: {accuracy}")

            # print(f"Classification Report: {classification_report(y_pred, y_test)}")

            return accuracy
        
        processedTrain = [self.advancedPreprocessing(x) for x in X_train]
        processedTest = [self.advancedPreprocessing(x) for x in X_test]

        self.model.fit(processedTrain, y_train)

        y_pred = self.model.predict(processedTest)

        accuracy = accuracy_score(y_test, y_pred)
        # print(f"The accuracy is: {accuracy}")

        classReport = classification_report(y_test, y_pred)
        # print(f"The classification report is: {classReport}")

        val_score = cross_val_score(self.model, processedTrain, y_train, cv=5)
        # print(f"The cross validation scores: {val_score.mean():2f}\n{val_score.std():2f}")

        return accuracy
    
    def predictQuery(self, query: str) -> Tuple[str, float]:
        if self.method == "hybrid" and self.model and self.vectorizer:
            processed_query = self.advancedPreprocessing(query)
            query_vec = self.vectorizer.transform([processed_query])
    
            final_intent = self.model.predict(query_vec)[0]
            intent_probability = self.model.predict_proba(query_vec)[0]
    
            confidence = max(intent_probability)
        
        else:
            processed_query = self.advancedPreprocessing(query)
            # Pass as a list of strings if using pipeline
            final_intent = self.model.predict([processed_query])[0]
    
            if hasattr(self.model, "predict_proba"):
                intent_probability = self.model.predict_proba([processed_query])[0]
                confidence = max(intent_probability)
            else:
                confidence = 0.8
    
        features = self.extract_advanced_features(query)
    
        if final_intent == "personal" and features['personal_pronoun_count'] > 0:  
            confidence = min(confidence + 0.1, 1.0)
        elif final_intent == "general" and features['question_word_count'] > 0:
            confidence = min(confidence + 0.1, 1.0)
    
        return final_intent, confidence
    
    def saveModel(self, filepath = "intentClassifier.pkl"):
        model_data = {
            'model': self.model,
            'method': self.method,
            'vectorizer': self.vectorizer if hasattr(self, 'vectorizer') else None
        }
        joblib.dump(model_data, filepath)
        print(f"Model saved to {filepath}")

    def loadModel(self, filepath="intentClassifier.pkl"):
        try:
            model_data = joblib.load(filepath)
            self.model = model_data['model']
            self.method = model_data['method']
            self.vectorizer = model_data['vectorizer'] if model_data['vectorizer'] else None
            print(f"Model loaded from {filepath}")
        except FileNotFoundError:
            print(f'{filepath} was not found!!')


In [24]:
def benchmark_models():
    models = ["hybrid", "naive_bayes", "random_forest"]
    results = {}

    for model in models:
        print(f"\n{'='*50}")
        lwc = lightWeightIntentClassifier(method=model)
        accuracy = lwc.train_model()
        results[model] = accuracy

    print(f"\n{'='*50}")
    print("Prediction Results: ")
    print(f"\n{'='*50}")

    for method, accuracy in results.items():
        print(f'For {method}: {accuracy}')

In [25]:
def test_classifier():
    lwc = lightWeightIntentClassifier(method="hybrid")
    lwc.train_model()

    test_queries = [
        "How much did I spend on groceries this month?",
        "What is compound interest?",
        "Show me my recent transactions",
        "How do mutual funds work?",
        "What's my budget status?",
        "Explain stock market basics",
        "My credit card bill amount",
        "Best investment strategies"
    ]

    print(f"\n{'='*50}")
    print("Testing queries: ")
    print(f"\n{'='*50}")

    for query in test_queries:
        final_intent, confidence = lwc.predictQuery(query)

        print(f'Query: {query}\n')
        print(f'Category: {final_intent}, Confidence: {confidence}\n\n')

In [ ]:
benchmark_models()
test_classifier()





Prediction Results: 

For hybrid: 0.8
For naive_bayes: 0.8
For random_forest: 0.8

Testing queries: 

Query: How much did I spend on groceries this month?

Category: personal, Confidence: 1.0


Query: What is compound interest?

Category: general, Confidence: 0.9486588329248901


Query: Show me my recent transactions

Category: personal, Confidence: 0.9694494569172843


Query: How do mutual funds work?

Category: general, Confidence: 1.0


Query: What's my budget status?

Category: personal, Confidence: 1.0


Query: Explain stock market basics

Category: general, Confidence: 0.9878430885039857


Query: My credit card bill amount

Category: personal, Confidence: 0.9850355567518896


Query: Best investment strategies

Category: general, Confidence: 0.634267463180672




In [ ]:
lwc = lightWeightIntentClassifier(method="hybrid")
lwc.train_model()
print(lwc.predictQuery("Which is the best bank for sip?"))

(np.str_('personal_sql'), np.float64(0.5100091138627669))


In [ ]:
lwc.saveModel()

Model saved to intentClassifier.pkl
